# Student Placement Eligibility Prediction

## Objective
Build a binary classification model using **Logistic Regression** to predict whether a student is likely to meet the placement eligibility criterion.

### Predictors
- `cgpa`
- `attendance_pct`
- `coding_score`
- `projects_completed`
- `internship_months`
- `backlogs`

### Target
- `target = 1` → Positive / event class
- `target = 0` → Negative / non-event class

### Evaluation Metrics
Accuracy, Precision, Recall, F1-score, ROC-AUC and Confusion Matrix.


In [ ]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve
)

print("Libraries imported successfully.")


## 2. Load Dataset

The notebook first tries the assignment filename. If it is not found, it uses the uploaded CSV path.


In [ ]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

import os

file_path = "dataset_07_student_placement_eligibility.csv"

# Fallback for the uploaded file in this environment
if not os.path.exists(file_path):
    file_path = "/mnt/data/94c9ff2f-3770-47ff-b224-64859e796b5d.csv"

df = pd.read_csv(file_path)

print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)
print("File:", file_path)
print("Shape:", df.shape)

display(df.head())


## 3. Dataset Inspection

In [ ]:
# ============================================================
# 3. DATASET INSPECTION
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nStatistical summary:")
display(df.describe())

print("\nDataset information:")
df.info()


## 4. Missing Values, Duplicates and Data Quality

In [ ]:
# ============================================================
# 4. DATA QUALITY CHECKS
# ============================================================

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_values = df.isnull().sum()
display(missing_values.to_frame("Missing Values"))

print("Total missing values:", missing_values.sum())

print("\n" + "=" * 60)
print("DUPLICATE CHECK")
print("=" * 60)

duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

print("\n" + "=" * 60)
print("TARGET CLASS BALANCE")
print("=" * 60)

target_counts = df["target"].value_counts().sort_index()
target_percent = df["target"].value_counts(normalize=True).sort_index() * 100

balance_df = pd.DataFrame({
    "Count": target_counts,
    "Percentage": target_percent.round(2)
})

display(balance_df)


In [ ]:
# ============================================================
# 5. DATA RANGE / VALIDITY CHECK
# ============================================================

features = [
    "cgpa",
    "attendance_pct",
    "coding_score",
    "projects_completed",
    "internship_months",
    "backlogs"
]

print("=" * 60)
print("FEATURE RANGES")
print("=" * 60)

range_df = pd.DataFrame({
    "Minimum": df[features].min(),
    "Maximum": df[features].max(),
    "Mean": df[features].mean(),
    "Unique Values": df[features].nunique()
})

display(range_df)


In [ ]:
# ============================================================
# 6. TARGET DISTRIBUTION PLOT
# ============================================================

plt.figure(figsize=(6, 4))

df["target"].value_counts().sort_index().plot(kind="bar")

plt.title("Target Class Distribution")
plt.xlabel("Target")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


## 5. Define Features and Target

No target information is used during preprocessing. This prevents target leakage.


In [ ]:
# ============================================================
# 7. DEFINE FEATURES AND TARGET
# ============================================================

X = df[features].copy()
y = df["target"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())


## 6. Train/Test Split

An **80/20 stratified split** is used. Stratification preserves the class distribution in both training and test sets.


In [ ]:
# ============================================================
# 8. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining class distribution:")
display(y_train.value_counts().sort_index())

print("\nTesting class distribution:")
display(y_test.value_counts().sort_index())


## 7. Logistic Regression Pipeline

`StandardScaler` is included because the predictors have different numerical scales.

The scaler is inside a `Pipeline`, so it is fitted only on the training data. This avoids data leakage from the test set.


In [ ]:
# ============================================================
# 9. LOGISTIC REGRESSION MODEL
# ============================================================

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

print(model)


In [ ]:
# ============================================================
# 10. TRAIN MODEL
# ============================================================

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")


In [ ]:
# ============================================================
# 11. PREDICTIONS
# ============================================================

y_pred = model.predict(X_test)

# Probability of target = 1
y_prob = model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")


## 8. Model Evaluation

In [ ]:
# ============================================================
# 12. PERFORMANCE METRICS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

results["Score"] = results["Score"].round(4)

display(results)


In [ ]:
# ============================================================
# 13. CLASSIFICATION REPORT
# ============================================================

print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "Not Eligible / Negative",
        "Eligible / Positive"
    ],
    zero_division=0
))


## 9. Confusion Matrix

In [ ]:
# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nInterpretation:")
print("True Negatives  (TN):", tn)
print("False Positives (FP):", fp)
print("False Negatives (FN):", fn)
print("True Positives  (TP):", tp)


In [ ]:
# Plot confusion matrix

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Eligible", "Eligible"]
)

disp.plot(ax=ax)

ax.set_title("Confusion Matrix - Logistic Regression")

plt.tight_layout()
plt.show()


## 10. ROC Curve

In [ ]:
# ============================================================
# 15. ROC CURVE
# ============================================================

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.tight_layout()
plt.show()


## 11. Coefficient and Feature-Effect Analysis

Because the model uses standardized features, each coefficient represents the change in log-odds of `target = 1` associated with a one-standard-deviation increase in that feature, holding the other features constant.

- Positive coefficient → increases the odds of `target = 1`.
- Negative coefficient → decreases the odds of `target = 1`.


In [ ]:
# ============================================================
# 16. LOGISTIC REGRESSION COEFFICIENTS
# ============================================================

logistic_model = model.named_steps["logistic_regression"]

coefficients = logistic_model.coef_[0]

coefficient_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": coefficients
})

coefficient_df["Absolute_Coefficient"] = (
    coefficient_df["Coefficient"].abs()
)

coefficient_df = coefficient_df.sort_values(
    "Absolute_Coefficient",
    ascending=False
)

display(
    coefficient_df[
        ["Feature", "Coefficient", "Absolute_Coefficient"]
    ].round(4)
)


In [ ]:
# ============================================================
# 17. ODDS RATIOS
# ============================================================

coefficient_df["Odds_Ratio"] = np.exp(
    coefficient_df["Coefficient"]
)

display(
    coefficient_df[
        ["Feature", "Coefficient", "Odds_Ratio"]
    ].round(4)
)


In [ ]:
# ============================================================
# 18. AUTOMATIC FEATURE INTERPRETATION
# ============================================================

print("=" * 60)
print("FEATURE EFFECT INTERPRETATION")
print("=" * 60)

for _, row in coefficient_df.iterrows():

    feature = row["Feature"]
    coefficient = row["Coefficient"]
    odds_ratio = row["Odds_Ratio"]

    if coefficient > 0:
        direction = "positive"
        effect = "increases"
    elif coefficient < 0:
        direction = "negative"
        effect = "decreases"
    else:
        direction = "neutral"
        effect = "has little/no"

    print(
        f"{feature}: {direction} effect; "
        f"coefficient = {coefficient:.4f}, "
        f"odds ratio = {odds_ratio:.4f}. "
        f"A one-standard-deviation increase {effect} "
        f"the odds of target=1."
    )


In [ ]:
# ============================================================
# 19. COEFFICIENT VISUALIZATION
# ============================================================

plot_df = coefficient_df.sort_values("Coefficient")

plt.figure(figsize=(8, 5))

plt.barh(
    plot_df["Feature"],
    plot_df["Coefficient"]
)

plt.axvline(
    x=0,
    linestyle="--"
)

plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Feature Effects on Placement Eligibility")

plt.tight_layout()
plt.show()


## 12. Example Student Prediction

The following example shows how the trained model can be used to predict a new student's placement eligibility.


In [ ]:
# ============================================================
# 20. NEW STUDENT PREDICTION
# ============================================================

new_student = pd.DataFrame({
    "cgpa": [8.2],
    "attendance_pct": [85],
    "coding_score": [8],
    "projects_completed": [4],
    "internship_months": [6],
    "backlogs": [0]
})

prediction = model.predict(new_student)[0]
probability = model.predict_proba(new_student)[0, 1]

print("Student details:")
display(new_student)

print("Predicted class:", prediction)
print(f"Probability of target=1: {probability:.4f}")

if prediction == 1:
    print("Prediction: Likely to meet placement eligibility.")
else:
    print("Prediction: Likely not to meet placement eligibility.")


## 13. Final Conclusion

The Logistic Regression model provides a simple and interpretable baseline for student placement eligibility prediction.

The final evaluation should be based on:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix

### Limitations
1. The dataset contains only six predictors.
2. Placement eligibility may depend on additional factors such as communication skills, aptitude scores, interview performance and company-specific criteria.
3. A single train/test split can produce performance estimates that vary depending on the random split.
4. The model assumes a linear relationship between the predictors and the log-odds of the target.
5. The dataset may not represent students from different colleges, academic years or recruitment environments.

### Possible Improvements
- Use cross-validation for more robust performance estimates.
- Collect a larger and more diverse dataset.
- Add relevant academic and placement-related features.
- Tune the classification threshold depending on whether false negatives or false positives are more costly.
- Compare Logistic Regression with other models only if technically justified.


In [ ]:
# ============================================================
# 21. FINAL RESULTS SUMMARY
# ============================================================

print("=" * 60)
print("FINAL MODEL RESULTS")
print("=" * 60)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nModel Configuration:")
print("Algorithm       : Logistic Regression")
print("Preprocessing   : StandardScaler")
print("Test Size       : 20%")
print("Random State    : 42")
print("Stratification  : Yes")
print("Max Iterations  : 1000")
